# PyTorch MEMO Example

This notebook shows how to run a single image with the native PyTorch MEMO model and display the results inline.

The example image is downloaded automatically from a public URL, so this notebook does not require a local edge_data folder.

## What This Notebook Does

- Lists the available MEMO checkpoints
- Lists the downloadable public example images
- Downloads one example image into experiments/demo_examples
- Runs the same native PyTorch inference path used by the repository
- Displays the original image, prediction map, and binarized result

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_example_assets import ensure_demo_example, list_demo_examples
from demo_model_registry import list_model_presets, resolve_model_preset
from deployment.memo_runtime import OptimizedMEMOPredictor

In [ ]:
available_models = {name: preset['description'] for name, preset in list_model_presets().items()}
available_examples = {name: metadata['description'] for name, metadata in list_demo_examples().items()}
{'models': available_models, 'examples': available_examples}

In [ ]:
config = {
    'model_name': 'Synthetic Late',
    'example_name': 'Lena',
    'guidance_scale': 1.4,
    'max_steps': 20,
    'resize_long_side': 320,
    'device': 'auto',
    'precision': 'fp16',
    'compile_model': False,
}

config['image_path'] = ensure_demo_example(config['example_name'])
config

In [ ]:
preset = resolve_model_preset(config['model_name'])
predictor = OptimizedMEMOPredictor(
    config_file=preset['config_file'],
    model_path=preset['model_path'],
    base_model_path=preset['base_model_path'],
    device=config['device'],
    precision=config['precision'],
    enable_compile=config['compile_model'],
)

image_bgr = cv2.imread(str(config['image_path']), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise FileNotFoundError(config['image_path'])

result = predictor.predict_bgr(
    image_bgr,
    guidance_scale=config['guidance_scale'],
    max_steps=config['max_steps'],
    resize_long_side=config['resize_long_side'],
)

original_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
prediction_rgb = cv2.cvtColor(result['prediction'], cv2.COLOR_GRAY2RGB)
binarized_rgb = cv2.cvtColor(result['binarized'], cv2.COLOR_GRAY2RGB)

print('model:', config['model_name'])
print('original shape:', original_rgb.shape)
print('prediction shape:', result['prediction'].shape)
print('binarized shape:', result['binarized'].shape)

In [ ]:
def add_title_bar(image_rgb: np.ndarray, title: str, bar_height: int = 28) -> np.ndarray:
    canvas = np.full((image_rgb.shape[0] + bar_height, image_rgb.shape[1], 3), 255, dtype=np.uint8)
    canvas[bar_height:] = image_rgb
    cv2.putText(
        canvas,
        title,
        (8, 19),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (20, 20, 20),
        1,
        cv2.LINE_AA,
    )
    return canvas

panels = [
    add_title_bar(original_rgb, 'Original'),
    add_title_bar(prediction_rgb, 'Prediction'),
    add_title_bar(binarized_rgb, 'Binarized'),
]

max_height = max(panel.shape[0] for panel in panels)
normalized_panels = []
for panel in panels:
    if panel.shape[0] < max_height:
        pad = np.full((max_height - panel.shape[0], panel.shape[1], 3), 255, dtype=np.uint8)
        panel = np.concatenate([panel, pad], axis=0)
    normalized_panels.append(panel)

preview = np.concatenate(normalized_panels, axis=1)
display(Image.fromarray(preview))

In [ ]:
save_dir = PROJECT_ROOT / 'experiments' / 'pytorch_notebook_demo'
save_dir.mkdir(parents=True, exist_ok=True)

cv2.imwrite(str(save_dir / 'example_prediction.png'), result['prediction'])
cv2.imwrite(str(save_dir / 'example_binarized.png'), result['binarized'])

print('saved to:', save_dir)